## Data Processing Pipeline

Level 1: Raw forecast, raw analysis \
Level 2: Forecast and analysis anomalies \
Level 3: Secondary derived terms--squared anomalies, covariance \
Level 4: Final metrics--MSE, ACC \
Level 5: Aggregated quantities

Scoping: Compute full grid for 1 year (2020), then aggregate all tracts (stress test)

# County-level forecast error map

This section visualizes **forecast vs observed 2m temperature errors** at the **county level** using an interactive Plotly choropleth.

You will be able to:
- Select a **date/time** and view the forecast error for each county.
- Hover over counties to inspect **forecast**, **observed**, **error**, and basic identifying information (FIPS, county name, state).

The code below assumes a precomputed, county-level, time-indexed dataset of forecast and observed values (and their error) that can be loaded from disk.

In [ ]:
import os
import json
from datetime import datetime

import pandas as pd
import plotly.express as px
import plotly.io as pio

# Optional: widgets for interactive time selection
import ipywidgets as widgets
from IPython.display import display

# Use a renderer appropriate for your environment (can be changed as needed)
pio.renderers.default = "notebook_connected"  # e.g., "notebook_connected", "vscode", or "plotly_mimetype"

In [ ]:
# -----------------------------------------------------------------------------
# Load county-level forecast/observed error data
# -----------------------------------------------------------------------------

# Path to a precomputed, county-level, time-indexed error dataset.
# Expected minimal schema (column names can be adapted here if needed):
# - county_fips      : 5-character FIPS code for each county (string or int)
# - county_name      : human-readable county name
# - state            : state abbreviation or name
# - timestamp        : forecast valid time (ISO string or datetime-like)
# - forecast_value   : forecast 2m temperature (K or degC, consistent with observed_value)
# - observed_value   : observed/analysis 2m temperature
# - error            : forecast_value - observed_value (can be recomputed below if needed)

ERROR_DATA_PATH = "/path/to/county_level_2t_error.parquet"  # TODO: set this to your dataset

if ERROR_DATA_PATH.endswith(".parquet"):
    df = pd.read_parquet(ERROR_DATA_PATH)
elif ERROR_DATA_PATH.endswith(".csv"):
    df = pd.read_csv(ERROR_DATA_PATH)
else:
    raise ValueError("ERROR_DATA_PATH must be a .parquet or .csv file. Please update the extension or loading logic.")

# Standardize and derive helpful columns
if "county_fips" not in df.columns:
    raise KeyError("Expected a 'county_fips' column in the error dataset. Please adapt this cell to your schema.")

# Ensure county_fips are zero-padded 5-character strings
df["county_fips"] = df["county_fips"].astype(str).str.zfill(5)

# Ensure we have a timestamp column and convert to datetime
if "timestamp" not in df.columns:
    raise KeyError("Expected a 'timestamp' column in the error dataset. Please adapt this cell to your schema.")

df["timestamp"] = pd.to_datetime(df["timestamp"])

# If error is not present, compute it
if "error" not in df.columns:
    if {"forecast_value", "observed_value"}.issubset(df.columns):
        df["error"] = df["forecast_value"] - df["observed_value"]
    else:
        raise KeyError("Dataset is missing 'error' and also 'forecast_value'/'observed_value'. Please supply at least one of these.")

# Convenience date/hour fields for filtering or grouping
df["date"] = df["timestamp"].dt.date
df["hour"] = df["timestamp"].dt.hour

# Quick sanity check on the loaded data
df.head()

In [ ]:
# -----------------------------------------------------------------------------
# Load county GeoJSON and align keys
# -----------------------------------------------------------------------------

# This cell expects a GeoJSON file with US counties (or your region of interest).
# The GeoJSON must contain a property that matches the county FIPS codes used
# in df["county_fips"]. Commonly this property is named "GEOID".

GEOJSON_PATH = "/path/to/us_counties.geojson"  # TODO: set this to your counties GeoJSON
GEOJSON_FIPS_PROPERTY = "GEOID"  # Update if your GeoJSON uses a different property name

with open(GEOJSON_PATH, "r") as f:
    counties_geojson = json.load(f)

# Optionally, you can perform a quick check that a sample FIPS exists
sample_fips = df["county_fips"].iloc[0]
geojson_fips_values = {
    feature["properties"].get(GEOJSON_FIPS_PROPERTY)
    for feature in counties_geojson["features"]
}

if sample_fips not in geojson_fips_values:
    print(
        f"Warning: sample county_fips {sample_fips} not found in GeoJSON property '" \
        f"{GEOJSON_FIPS_PROPERTY}'. You may need to adjust GEOJSON_FIPS_PROPERTY or county_fips formatting."
    )
else:
    print(f"Sample county_fips {sample_fips} found in GeoJSON; keys appear to be aligned.")

In [ ]:
# -----------------------------------------------------------------------------
# Plotly choropleth: county-level forecast error for a given time
# -----------------------------------------------------------------------------

from typing import Optional


def make_error_map(df_slice: pd.DataFrame, timestamp_label: Optional[str] = None):
    """Return a Plotly choropleth of forecast error by county for a given time slice.

    Parameters
    ----------
    df_slice : pandas.DataFrame
        A DataFrame filtered to a single timestamp, containing at least:
        - county_fips
        - error
        - county_name (optional, for hover)
        - state (optional, for hover)
        - forecast_value, observed_value (optional, for hover)
    timestamp_label : str, optional
        A string to include in the map title (e.g., the timestamp).
    """
    if df_slice.empty:
        raise ValueError("df_slice is empty; check that the selected timestamp exists in the data.")

    # Use a symmetric color range around zero so over/under prediction is clear
    max_abs_error = float(df_slice["error"].abs().max())
    if max_abs_error == 0:
        max_abs_error = 1.0  # avoid zero range

    hover_data = {
        "county_fips": True,
        "error": ':.2f',
    }
    if "county_name" in df_slice.columns:
        hover_data["county_name"] = True
    if "state" in df_slice.columns:
        hover_data["state"] = True
    if "forecast_value" in df_slice.columns:
        hover_data["forecast_value"] = ':.2f'
    if "observed_value" in df_slice.columns:
        hover_data["observed_value"] = ':.2f'

    title = "County-level forecast error"
    if timestamp_label is not None:
        title += f" — {timestamp_label}"

    fig = px.choropleth(
        df_slice,
        geojson=counties_geojson,
        locations="county_fips",
        featureidkey=f"properties.{GEOJSON_FIPS_PROPERTY}",
        color="error",
        color_continuous_scale="RdBu_r",
        range_color=(-max_abs_error, max_abs_error),
        hover_data=hover_data,
        labels={"error": "Forecast − Observed"},
        title=title,
    )

    fig.update_geos(fitbounds="locations", visible=False)
    fig.update_layout(margin={"r": 0, "t": 50, "l": 0, "b": 0})
    return fig


def plot_for_timestamp(ts) -> None:
    """Helper to filter the main DataFrame to a single timestamp and show the map."""
    ts = pd.to_datetime(ts)
    df_ts = df[df["timestamp"] == ts]
    fig = make_error_map(df_ts, timestamp_label=str(ts))
    return fig

In [ ]:
# -----------------------------------------------------------------------------
# Interactive time selection (date/time) with ipywidgets
# -----------------------------------------------------------------------------

# Unique sorted timestamps for selection
unique_times = sorted(df["timestamp"].unique())

if not unique_times:
    raise ValueError("No timestamps found in df['timestamp']. Ensure the dataset was loaded correctly.")

# A slider (or dropdown) to select a timestamp
# If you have many timestamps, you may prefer a SelectionSlider; for fewer, a Dropdown is fine.
time_slider = widgets.SelectionSlider(
    options=unique_times,
    index=0,
    description="Valid time",
    continuous_update=False,
    layout=widgets.Layout(width="80%"),
)

output = widgets.Output()


def _update_map(change):
    if change["name"] != "value":
        return
    with output:
        output.clear_output(wait=True)
        fig = plot_for_timestamp(change["new"])
        fig.show()


# Wire the callback and display the controls
_time_obs = time_slider.observe(_update_map, names="value")

display(time_slider)

# Initial render
with output:
    fig0 = plot_for_timestamp(unique_times[0])
    fig0.show()

display(output)

In [ ]:
# -----------------------------------------------------------------------------
# Additional sanity checks and quick diagnostics
# -----------------------------------------------------------------------------

# Distribution of errors for a sample time (first timestamp)
sample_ts = unique_times[0]
df_sample = df[df["timestamp"] == sample_ts]

print(f"Sample timestamp: {sample_ts}")
print("Number of counties:", len(df_sample))
print("Error summary (K or chosen units):")
print(df_sample["error"].describe())

# Optional: simple histogram of errors (requires plotly or matplotlib). Using Plotly here.
fig_hist = px.histogram(df_sample, x="error", nbins=40, title=f"Error distribution — {sample_ts}")
fig_hist.update_layout(xaxis_title="Forecast − Observed", yaxis_title="Count")
fig_hist.show()